# A LoKI batch: the UI

The same client API as `loki-batch.ipynb`, driven by clicking. One batch, one
table: rows come from a dataset picker or from the batch's selector, editing a
cell pins it, and one Reduce button runs every row that is not up to date.
Hover over a control for what it does.

## Setup

In [ ]:
import io
import tempfile
from pathlib import Path

import ipywidgets as widgets
import pandas as pd
import plopp as pp
from IPython.display import display

from ess.apps import loki
from ess.apps.batch import TriggerLoop, apply, batch_table, dataset_table
from ess.apps.client import local
from ess.apps.records import Status
from ess.apps.rules import AsOf, Bound, Like, Lookup, LookupEntry, Rule, Selector, Template
from ess.apps.sources import FolderSource
from ess.apps.spec import as_ref, dataset_ref
from ess.reduce.spec.parameters import QEdges

journal = pd.DataFrame.from_records(
    [
        (60384, 'porous silica', 'transmission'),
        (60385, 'porous silica', 'sample'),
        (60386, 'AgBeh', 'transmission'),
        (60387, 'AgBeh', 'sample'),
        (60388, 'deuterated SDS', 'transmission'),
        (60389, 'deuterated SDS', 'sample'),
        (60392, 'empty beam', 'empty-beam'),
        (60393, 'solvent', 'background'),
        (60394, 'ISIS polymer', 'transmission'),
        (60395, 'ISIS polymer', 'sample'),
    ],
    columns=['run', 'sample', 'role'],
).set_index('run')

In [ ]:
cache = loki.cache()
root = Path(tempfile.mkdtemp(prefix='loki-batch-ui-'))
incoming = root / 'incoming'
incoming.mkdir()


def arrive(*runs):
    """Let runs arrive: link their tutorial files into the folder the source reads."""
    for run in runs:
        (path,) = cache.glob(f'{run}-*.nxs')
        (incoming / path.name).symlink_to(path)


arrive(*journal.index.drop([60394, 60395]))

client = local(
    root / 'store',
    instrument='loki',
    proposal='p1',
    submitter='notebook',
    registry=loki.registry(),
    sources=[
        FolderSource(
            incoming,
            '*.nxs',
            identity=r'(?P<run>\d+)-.*',
            instrument='loki',
            journal=journal.to_dict('index'),
        )
    ],
)


def run_ref(number):
    return dataset_ref(instrument='loki', run=number)


center = client.run(loki.BEAM_CENTER, {'sample_run': run_ref(60387)})
template = Template(
    name='loki-iofq-larmor',
    spec=loki.IOFQ.id,
    params={
        'background_run': run_ref(60393),
        'background_transmission_run': run_ref(60392),
        'empty_beam_run': run_ref(60392),
        'direct_beam': dataset_ref(path=cache / 'direct-beam-loki-all-pixels.h5'),
        'beam_center': center.ref(),
        'q': QEdges(start=0.01, stop=0.3, num_bins=100),
    },
    blanks=('sample_run', 'sample_transmission_run'),
    dataset_field='sample_run',
)
transmission_lookup = Lookup(
    name='transmission',
    entries=(
        LookupEntry(
            name='nearest-transmission',
            fills={
                'sample_transmission_run': AsOf(match={'role': Like(pattern='transmission')})
            },
        ),
    ),
)

## The app

The batch is a rule: the template's defaults, the transmission lookup, and a
selector with a bound. Auto-reduce is the rule being active. Every button is
`apply` on that rule; the row status says why a row would run again.

In [ ]:
state = {
    'rule': Rule(
        name='loki-iofq',
        template=template,
        lookup=transmission_lookup,
        selector=Selector(match={'role': Like(pattern='sample')}, after=Bound(run=60393)),
        active=False,
    )
}
state['loop'] = TriggerLoop(client, state['rule'])
rows = {}
NEEDS_RUN = {'not reduced', 'edited', 'defaults changed', 'failed', 'cancelled'}


def datasets():
    return sorted(client.datasets(), key=lambda d: d.run or 0)


def sample_name(dataset):
    return dataset.fields.get('sample', str(dataset.ref))


def options_for(role):
    """Dropdown options: run number and sample name to a reference."""
    return [(f'{d.run} {sample_name(d)}', d.ref) for d in datasets() if d.fields['role'] == role]


def help_text(text):
    return widgets.HTML(f'<span style="color: #666">{text}</span>')


def html_table(frame):
    return widgets.HTML(frame.to_html())


# ---- Settings: the template's defaults and the selector. A change is a new version.

q_start_w = widgets.FloatText(
    value=0.01, description='Q start', tooltip='Default for every row that does not pin its own Q.'
)
q_stop_w = widgets.FloatText(value=0.3, description='Q stop', tooltip=q_start_w.tooltip)
q_bins_w = widgets.IntText(value=100, description='Q bins', tooltip=q_start_w.tooltip)
role_w = widgets.Dropdown(
    options=sorted({d.fields['role'] for d in datasets()}),
    value='sample',
    description='select role',
    tooltip='Every dataset with this journal role gets a row in the batch.',
)
bound_w = widgets.IntText(
    value=60393,
    description='auto after run',
    tooltip=(
        'With auto-reduce on, selected datasets after this run are reduced as they arrive. '
        'Earlier ones stay "not reduced" until you press Reduce: the backlog.'
    ),
)
auto_w = widgets.Checkbox(
    value=False,
    description='auto-reduce',
    indent=False,
    tooltip='Let the scheduler reduce selected datasets after the bound as they arrive.',
)
save_button = widgets.Button(
    description='Save as new version',
    tooltip=(
        'Store the defaults and the selector as a new version. Rows reduced with an older '
        'version show "defaults changed"; press Reduce to reprocess them with their pins kept.'
    ),
)
version_w = widgets.Label()


def save_version(_=None):
    """A new rule version from the form; rows made by the old one show 'defaults changed'."""
    rule = state['rule']
    q = QEdges(start=q_start_w.value, stop=q_stop_w.value, num_bins=q_bins_w.value)
    selector = Selector(match={'role': Like(pattern=role_w.value)}, after=Bound(run=bound_w.value))
    state['rule'] = rule.revise(template=rule.template.revise(q=q), selector=selector)
    state['loop'] = TriggerLoop(client, state['rule'])
    refresh()


def set_auto(change):
    """Auto-reduce is the rule being active: mutable state, not a version."""
    state['rule'].active = change['new']
    refresh()


# ---- Rows: one per dataset the batch knows, with the cells a person may pin.


def fills(dataset):
    """What the rule fills for a dataset without pins: here the transmission run."""
    try:
        request = apply(client, state['rule'], [dataset])[str(dataset.ref)]
    except ValueError:
        return None
    return as_ref(request.inputs['sample_transmission_run'])


def make_row(dataset, record=None):
    """An editable row whose cells start from the record's pins, else the rule's fills."""
    params = record.request.values() if record is not None else {}
    pinned = set(record.request.origin.pinned) if record is not None else set()
    default = fills(dataset)
    q = QEdges.model_validate(params['q']) if 'q' in pinned else None
    transmission = (
        as_ref(params['sample_transmission_run']) if 'sample_transmission_run' in pinned else default
    )
    q_tip = "This row's own Q binning; the defaults above no longer apply to it."
    row = {
        'dataset': dataset,
        'default': default,
        'transmission': widgets.Dropdown(
            options=options_for('transmission'),
            value=transmission,
            tooltip=(
                'Transmission run, filled by the lookup: the nearest one before the sample. '
                'Choosing another pins it for this row.'
            ),
        ),
        'pin_q': widgets.Checkbox(value=q is not None, description='pin Q', indent=False, tooltip=q_tip),
        'q_start': widgets.FloatText(value=q.start if q else q_start_w.value, description='start'),
        'q_stop': widgets.FloatText(value=q.stop if q else q_stop_w.value, description='stop'),
        'q_bins': widgets.IntText(value=q.num_bins if q else q_bins_w.value, description='bins'),
        'status': widgets.Label(layout=widgets.Layout(width='130px')),
        'remove': widgets.Button(
            layout=widgets.Layout(width='70px'),
            tooltip='Exclude this dataset from the batch; the scheduler skips it too. Restore undoes it.',
        ),
    }
    row['remove'].on_click(toggle_exclusion(str(dataset.ref)))
    return row


def pins(row):
    """The cells that differ from what the rule fills: what the request pins."""
    pinned = {}
    if row['transmission'].value != row['default']:
        pinned['sample_transmission_run'] = row['transmission'].value
    if row['pin_q'].value:
        pinned['q'] = QEdges(
            start=row['q_start'].value, stop=row['q_stop'].value, num_bins=row['q_bins'].value
        )
    return pinned


def desired(key, row):
    """The request the row asks for now: the rule applied to its dataset, plus its pins."""
    return apply(client, state['rule'], [row['dataset']], {key: pins(row)})[key]


def status(key, row):
    """Why the row would run again, or that it need not."""
    rule = state['rule']
    if key in rule.exclusions:
        return 'excluded'
    latest = client.latest(rule.name, member_key=key)
    if latest is None:
        return 'not reduced'
    if latest.status in (Status.FAILED, Status.CANCELLED):
        return latest.status.value
    if latest.status != Status.COMPLETED:
        return 'running'
    if latest.request.origin.rule != rule.id:
        return 'defaults changed'
    # The record also holds the spec's defaults, filled at submit, so only what
    # the row sets is compared.
    recorded = latest.request.values()
    if any(recorded.get(k) != v for k, v in desired(key, row).values().items()):
        return 'edited'
    return 'up to date'


def toggle_exclusion(key):
    def handler(_=None):
        rule = state['rule']
        if key in rule.exclusions:
            del rule.exclusions[key]
        else:
            rule.exclude(key, 'removed from the batch form')
        refresh()

    return handler


def row_box(key, row):
    name = widgets.Label(sample_name(row['dataset']), layout=widgets.Layout(width='110px'))
    dataset = widgets.Label(key, layout=widgets.Layout(width='90px'))
    cells = ['transmission', 'pin_q', 'q_start', 'q_stop', 'q_bins', 'status', 'remove']
    return widgets.HBox([name, dataset, *[row[cell] for cell in cells]])


def refresh_rows():
    """Rows are the batch's members, what the selector matches, and what was added by hand."""
    rule = state['rule']
    latest = {str(r.request.member_key): r for r in client.batch(rule.name)}
    for d in datasets():
        key = str(d.ref)
        if key not in rows and (key in latest or key in rule.exclusions or rule.selector.selects(d)):
            rows[key] = make_row(d, latest.get(key))
    for key, row in rows.items():
        row['status'].value = status(key, row)
        row['remove'].description = 'restore' if key in rule.exclusions else 'remove'
    order = sorted(rows, key=lambda key: rows[key]['dataset'].run or 0)
    rows_box.children = [row_box(key, rows[key]) for key in order]


rows_box = widgets.VBox([])
STATUS_LEGEND = (
    '<b>not reduced</b>: no record yet, Reduce runs it &middot; '
    '<b>up to date</b>: the latest record matches the row &middot; '
    '<b>edited</b>: a cell changed since, Reduce runs it again &middot; '
    '<b>defaults changed</b>: made by an older version, Reduce reprocesses it with its pins &middot; '
    '<b>excluded</b>: removed; neither Reduce nor the scheduler touches it'
)

# ---- Picker: add rows the selector does not.

pick_role_w = widgets.Dropdown(
    options=role_w.options, value='sample', description='role', tooltip='Which datasets the list offers.'
)
picker_w = widgets.SelectMultiple(
    description='datasets', rows=5, tooltip='Datasets the selector does not propose can be added here.'
)
add_button = widgets.Button(
    description='Add selected',
    tooltip='One row per selected dataset, filled like the selector would fill it.',
)


def refresh_picker(_=None):
    picker_w.options = [
        (f'{d.run}  {sample_name(d)}  {d.fields["role"]}', d)
        for d in datasets()
        if d.fields['role'] == pick_role_w.value
    ]


def add_selected(_=None):
    for d in picker_w.value:
        rows.setdefault(str(d.ref), make_row(d))
    refresh()


# ---- Reduce: every row that is not up to date, with what its cells say.

reduce_button = widgets.Button(
    description='Reduce',
    button_style='primary',
    tooltip=(
        'Run every row that is not up to date, with what its cells say. '
        'The backlog, an edited row, and rows made by an older version all run here.'
    ),
)
problems_box = widgets.VBox([])


def reduce_batch(_=None):
    """Backlog, a correction, and a reprocess are this one button in three situations."""
    group = {key: desired(key, row) for key, row in rows.items() if status(key, row) in NEEDS_RUN}
    reports = {key: client.validate(request) for key, request in group.items()}
    problems = {key: report.model_dump() for key, report in reports.items() if not report.ok}
    problems_box.children = [html_table(pd.DataFrame(problems).T)] if problems else []
    if problems:
        return
    client.submit_group(group)
    refresh()


# ---- Results: the batch table, the curves, and one member's history.

results_box = widgets.VBox([])
history_w = widgets.Dropdown(
    description='history of',
    tooltip='Every record made for this member: which version made it and what was pinned.',
)
history_box = widgets.VBox([])


def batch_with_samples():
    table = batch_table(client, state['rule']).join(dataset_table(client)['sample'])
    return table.set_index('sample', append=True)


def curves():
    return {
        sample_name(rows[str(r.request.member_key)]['dataset']): client.output(r, 'iofq')
        for r in client.batch(state['rule'].name)
        if r.status == Status.COMPLETED
    }


def plot_widget(figure):
    """A figure as a widget: plopp's own when it is one, else a PNG of it."""
    if isinstance(figure, widgets.Widget):
        return figure
    png = io.BytesIO()
    figure.save(png, format='png')
    return widgets.Image(value=png.getvalue())


def history(key):
    """Every record made for one member: which version made it, and what was pinned."""
    records = client.records(label=state['rule'].name, member_key=key)
    return pd.DataFrame(
        [
            {
                'record': r.id,
                'rule': r.request.origin.rule,
                'pinned': list(r.request.origin.pinned),
                'q_start': r.resolved_params['q']['start'],
                'q_bins': r.resolved_params['q']['num_bins'],
            }
            for r in records
        ]
    )


def refresh_history(_=None):
    history_box.children = [html_table(history(history_w.value))] if history_w.value else []


def refresh_results():
    table = batch_with_samples()
    if len(table):
        results_box.children = [html_table(table), plot_widget(pp.plot(curves(), norm='log'))]
    else:
        results_box.children = []
    history_w.options = [(sample_name(rows[key]['dataset']), key) for key in table.index.get_level_values(0)]
    refresh_history()


# ---- Instrument and scheduler, simulated.

arrive_button = widgets.Button(
    description='Arrive 60394, 60395',
    tooltip='Stands in for the instrument: the polymer and its transmission run are written.',
)
tick_button = widgets.Button(
    description='Scheduler tick',
    tooltip='Stands in for the scheduler, which does this every minute: fire the rule where it fires.',
)


def arrive_polymer(_=None):
    arrive(60394, 60395)
    refresh()


def tick(_=None):
    """What a scheduler would do every minute: fire the rule where it fires."""
    fired = state['loop'].run_once()
    refresh()
    return fired


# ---- The app.


def refresh():
    version_w.value = f'version {state["rule"].id}, auto-reduce {"on" if state["rule"].active else "off"}'
    refresh_rows()
    refresh_picker()
    refresh_results()


save_button.on_click(save_version)
auto_w.observe(set_auto, names='value')
pick_role_w.observe(refresh_picker, names='value')
add_button.on_click(add_selected)
reduce_button.on_click(reduce_batch)
history_w.observe(refresh_history, names='value')
arrive_button.on_click(arrive_polymer)
tick_button.on_click(tick)

app = widgets.VBox(
    [
        widgets.HTML('<b>Defaults and selector</b>'),
        help_text(
            'What every row gets unless it pins its own value, and which datasets get a row. '
            'Saving makes a new version; the old records keep saying which version made them.'
        ),
        widgets.HBox([q_start_w, q_stop_w, q_bins_w]),
        widgets.HBox([role_w, bound_w, auto_w, save_button, version_w]),
        widgets.HTML('<b>Batch</b>'),
        help_text(
            'One row per dataset. Edit a cell to pin it for that row, remove a row to exclude it, '
            'then press Reduce: it runs every row whose status says it needs to.'
        ),
        widgets.HBox([pick_role_w, picker_w, add_button]),
        rows_box,
        help_text(STATUS_LEGEND),
        reduce_button,
        problems_box,
        results_box,
        history_w,
        history_box,
        widgets.HTML('<b>Instrument and scheduler, simulated</b>'),
        help_text('Neither exists in this notebook; these two buttons stand in for them.'),
        widgets.HBox([arrive_button, tick_button]),
    ]
)
refresh()
display(app)